In [1]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt

from shapely.geometry import box

from skimage.registration import phase_cross_correlation

In [2]:
series_folder = Path(r".\data\test-series")
series_folder = Path(r"Z:\zeinab\ATLAS-projects\atlas_L32-10-1\atlas_L32-10-1_data\session_462189531\ROI-w1-20nm-bsd")
series_folder = Path(r"Z:\zeinab\ATLAS-projects\atlas_L32-10-1\atlas_L32-10-1_data\session_2053739366\ROI-w1-20nm-bsd")
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\atlas_L32-10-2\atlas_L32-10-2_20250410_data\session_955928929\w3-ROI-20nm-BSD")
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\atlas_L32-10-2\atlas_L32-10-2_20250410_data\session_1395343682\w2-ROI-20nm-BSD")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\TA31-luke_data\session_862126381\Section Set 3")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\TA31-luke_data\session_402716765\ta31-roi1-2")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA29\Site 3")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA30\ROI")
series_folder = Path(r'E:\PROJECTS\EM\zeinab\ATLAS-projects\atlas_olive_2\zeinab-olive-2-main_data\session_598049588\roi-01')
series_folder = Path(r'E:\PROJECTS\EM\Filipa\M2-2\whole sample sections 20251112_data\session_1077734160\Site 2')

series_folder = Path(r"E:\PROJECTS\EM\Filipa\M2-2\Filipa-M2-2-20251127\sections 8-12_data\session_1422202770\Site 1")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant-3-wafer-1\alma-plant-3_data\session_422002753\Site 2")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant2-wafer-1\plant 2_data\session_1115605826\Site 2")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant4-wafer-1\alma-plant-4_data\session_1084385689\Site 1")

series_folder = Path(r"E:\PROJECTS\EM\Filipa\M2-2\test\Site 2")

series_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        tif_files = list(folder.glob("*.tif"))
        if tif_files:  # Check if the list is not empty
            print(f"Found series folder: {folder.name} (contains {len(tif_files)} .tif files)")
            series_list.append(folder)

Found series folder: S_008_1788963903 (contains 9 .tif files)


In [3]:
from collections import defaultdict, deque
from atlas.stitching import get_tiles_dataframe, stitch_ATLAS_tiles
from atlas.stitching import add_tile_overlap_columns, match_tiles, build_adjacency_matrix_from_costs, build_transform_dict_from_mst, apply_transforms_and_stitch


In [4]:
from atlas.io import extract_s_number
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from collections import defaultdict, deque
import json

buffer_in_microns = 5
#TODO: Implement the max shift check
max_shift_in_pixles = 400

for raw_data_folder in series_list:
    # Check if files in the folder have the ".ve-tie" extension
    tie_file = None
    mif_file = None
    for file in raw_data_folder.iterdir():  # Iterate over all items in the folder
        #if file.is_file() and file.suffix == ".ve-tie":  # Check if it's a file with the desired extension
        #    print(f"File with '.ve-tie' extension found: {file.name}")
        #    tie_file = file
        if file.is_file() and file.suffix == ".ve-mif":  # Check if it's a file with the desired extension
            print(f"File with '.ve-mif' extension found: {file.name}")
            mif_file = file
            mif_tile_df = get_tiles_dataframe(mif_file, buffer_microns=buffer_in_microns)
            
            first_tif_path = raw_data_folder.joinpath(Path(mif_tile_df.iloc[0]['Filename']).name)
            extracted_number = extract_s_number(first_tif_path)
            # Define the output file path
            output_tif_path = raw_data_folder.parent.joinpath(f"stitched_image_S_{extracted_number}.tiff")
            output_cc_path = raw_data_folder.parent.joinpath(f"phaseCC_stitching_S_{extracted_number}.csv")
            output_jason_path = raw_data_folder.parent.joinpath(f"transforms_S_{extracted_number}.json")

            if output_tif_path.exists():
                print(f"✅ Skipping: {output_tif_path.name} already exists.")
            else:
                print(f"🔄 Stitching image for S_{extracted_number}...")
                #stitched_img, mif_tile_df = stitch_ATLAS_tiles(mif_tile_df, raw_data_folder, max_shift_pixels=max_shift_in_pixles)

                try:
                    mif_tile_df = add_tile_overlap_columns(mif_tile_df)
                    # add info to the DF so we know where to find the images after they have been moved out of the scope
                    mif_tile_df['raw_data_folder'] = raw_data_folder
                    
                    # Calculate the costs of matching each tile to those that it overlaps with
                    n = len(mif_tile_df)
                    all_costs = []
                    all_shifts = []

                    for current_idx in range(n):
                        # TODO: probably is at this level I have to add the shift max lim
                        costs, shifts = match_tiles(mif_tile_df, reference_idx=current_idx, min_overlap_percent = 2)
                        all_costs.append(costs)
                        all_shifts.append(shifts)

                    mif_tile_df["stitching_costs"] = all_costs
                    mif_tile_df["stitching_shifts"] = all_shifts


                    # Step 1: Build the cost matrix which I will use as adjacency for the min span tree
                    adj_matrix = build_adjacency_matrix_from_costs(mif_tile_df, cost_column='stitching_costs')

                    # Step 2: Create sparse matrix and compute MST
                    graph_sparse = csr_matrix(adj_matrix)
                    mst = minimum_spanning_tree(graph_sparse)
                    # The MST will be used to calculate the transofrmation matrices between each tile and a reference tile.
                    # For the moment I just pick 0 as reference but maybe there is a better way, in general I dont think it matters much.                
                    transform_dict = build_transform_dict_from_mst(mif_tile_df, mst, reference_tile=0)
                    
                    # apply transform, user inputs are transform_dict and mif_tile_df, output is the stitched_img
                    stitched_img = apply_transforms_and_stitch(mif_tile_df, transform_dict, reference_tile=0)



                    # Save the full image as a TIFF file
                    tiff.imwrite(output_tif_path, np.flipud(stitched_img))

                    mif_tile_df.to_csv(output_cc_path, index=False)

                    # Convert NumPy arrays to lists for JSON compatibility
                    json_ready_dict = {k: v.tolist() for k, v in transform_dict.items()}

                    # Save to JSON file
                    with open(output_jason_path, "w") as f:
                        json.dump(json_ready_dict, f, indent=2)
                        
                except Exception as e:
                    # Handle any error
                    print(f"Unexpected error with item {file}: {e}")
            
            


File with '.ve-mif' extension found: MosaicInfo_S_008_1788963903.ve-mif
🔄 Stitching image for S_8...

Processing reference tile 0...

Comparing reference 0 to tile 0...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 0 to tile 1...
Overlap %: 8
overlap img0 x: 3220-3500, y 0-3500
overlap img1 x: 0-280, y 0-3500
Detected pixel offset (row, col): [-22.  30.]

Comparing reference 0 to tile 2...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 0 to tile 3...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 0 to tile 4...
Overlap %: 1
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 0 to tile 5...
Overlap %: 8
overlap img0 x: 0-3500, y 0-280
overlap img1 x: 0-3500, y 3220-3500
Detected pixel offset (row, col): [-14.  16.]

Comparing reference 0 to tile 6...
Overlap %: 1
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 

In [ ]:
from atlas.stitching import get_total_canvas_size, get_overlap_relative, mask_low_and_saturation, first_last_true, extract_s_number

In [ ]:
raise old
# this is for me to check the max shift with currently is not implemented
from atlas.io import extract_s_number

buffer_in_microns = 40
max_shift_in_pixles = 400

for raw_data_folder in series_list:
    # Check if files in the folder have the ".ve-tie" extension
    tie_file = None
    mif_file = None
    for file in raw_data_folder.iterdir():  # Iterate over all items in the folder
        #if file.is_file() and file.suffix == ".ve-tie":  # Check if it's a file with the desired extension
        #    print(f"File with '.ve-tie' extension found: {file.name}")
        #    tie_file = file
        if file.is_file() and file.suffix == ".ve-mif":  # Check if it's a file with the desired extension
            print(f"File with '.ve-mif' extension found: {file.name}")
            mif_file = file

            mif_tile_df = get_tiles_dataframe(mif_file, buffer_microns=buffer_in_microns)

            #total_img_width, total_img_height = get_total_canvas_size(mif_tile_df)
            # check if stitching is already available
            first_tif_path = raw_data_folder.joinpath(Path(mif_tile_df.iloc[0]['Filename']).name)
            extracted_number = extract_s_number(first_tif_path)
            output_tif_path = raw_data_folder.parent.joinpath(f"stitched_image_S_{extracted_number}.tiff")
            if output_tif_path.exists():
                print(f"✅ Skipping: {output_tif_path.name} already exists.")
            else:
                print(f"🔄 Stitching image for S_{extracted_number}...")
                #stitched_img, mif_tile_df = stitch_ATLAS_tiles(mif_tile_df, raw_data_folder, max_shift_pixels=max_shift_in_pixles)

                try:
                    stitched_img, mif_tile_df = stitch_ATLAS_tiles(mif_tile_df, raw_data_folder, max_shift_pixels=400)
                except Exception as e:
                    # Handle any error
                    print(f"Unexpected error with item {file}: {e}")

            #fig, ax = plt.subplots(1, 1, figsize=(15, 15))
            #ax.imshow(stitched_img[:,:], cmap='gray')